# Lab 6

You are tasked with evaluating card counting strategies for black jack. In order to do so, you will use object oriented programming to create a playable casino style black jack game where a computer dealer plays against $n$ computer players and possibily one human player. If you don't know the rules of blackjack or card counting, please google it. 

A few requirements:
* The game should utilize multiple 52-card decks. Typically the game is played with 6 decks.
* Players should have chips.
* Dealer's actions are predefined by rules of the game (typically hit on 16). 
* The players should be aware of all shown cards so that they can count cards.
* Each player could have a different strategy.
* The system should allow you to play large numbers of games, study the outcomes, and compare average winnings per hand rate for different strategies.

1. Begin by creating a classes to represent cards and decks. The deck should support more than one 52-card set. The deck should allow you to shuffle and draw cards. Include a "plastic" card, placed randomly in the deck. Later, when the plastic card is dealt, shuffle the cards before the next deal.

In [ ]:
import random
from enum import Enum


# I’m using enums for Suit and Rank because they prevent typos
# and make the card representation more structured and readable.
class Suit(Enum):
    CLUBS = "C"
    DIAMONDS = "D"
    HEARTS = "H"
    SPADES = "S"


class Rank(Enum):
    TWO = "2"
    THREE = "3"
    FOUR = "4"
    FIVE = "5"
    SIX = "6"
    SEVEN = "7"
    EIGHT = "8"
    NINE = "9"
    TEN = "10"
    JACK = "J"
    QUEEN = "Q"
    KING = "K"
    ACE = "A"


class Card:
    # Each card stores its rank and suit.
    # I also added an "is_plastic" flag so the deck can include the cut card.
    def __init__(self, rank: Rank, suit: Suit, is_plastic: bool = False):
        self.rank = rank
        self.suit = suit
        self.is_plastic = is_plastic

    def __repr__(self):
        # The plastic card doesn’t have a rank or suit, so I display it differently.
        if self.is_plastic:
            return "<PLASTIC CARD>"
        return f"{self.rank.value}{self.suit.value}"


class Deck:
    """
    This class represents a blackjack shoe.
    A shoe contains multiple 52‑card decks (usually 6).
    I also insert a plastic cut card at a random position so the game knows
    when to reshuffle before the next round.
    """
    def __init__(self, num_decks: int = 6):
        self.num_decks = num_decks
        self.cards = []

        # Build the shoe with all the normal cards.
        self._build_shoe()

        # Insert the plastic card somewhere in the back half of the shoe.
        self._insert_plastic_card()

        # Shuffle the shoe after building it.
        self.shuffle()

    def _build_shoe(self):
        # I loop through each deck, then each suit, then each rank.
        # This creates num_decks * 52 total cards.
        for _ in range(self.num_decks):
            for suit in Suit:
                for rank in Rank:
                    self.cards.append(Card(rank, suit))

    def _insert_plastic_card(self):
        """
        The plastic card is placed somewhere near the back of the shoe.
        Casinos do this so the dealer knows when to reshuffle.
        I choose a random index between 50% and 90% of the shoe.
        """
        total = len(self.cards)
        start = total // 2
        end = int(total * 0.9)

        # Create the plastic card and insert it.
        plastic = Card(rank=None, suit=None, is_plastic=True)
        index = random.randint(start, end)
        self.cards.insert(index, plastic)

    def shuffle(self):
        """
        Shuffle the shoe, but keep the plastic card inside.
        After shuffling, I reinsert the plastic card at a new random position.
        This simulates how casinos reshuffle the shoe between rounds.
        """
        # Remove the plastic card temporarily.
        plastic_cards = [c for c in self.cards if c.is_plastic]
        self.cards = [c for c in self.cards if not c.is_plastic]

        # Shuffle the regular cards.
        random.shuffle(self.cards)

        # Reinsert exactly one plastic card.
        if not plastic_cards:
            plastic_cards = [Card(rank=None, suit=None, is_plastic=True)]

        self._insert_plastic_card()

    def draw_card(self):
        """
        Draw the top card from the shoe.
        I return both the card and a flag that tells the game
        whether the plastic card was dealt.
        """
        card = self.cards.pop(0)
        return card, card.is_plastic

    def cards_remaining(self):
        return len(self.cards)

    def __repr__(self):
        return f"Deck(num_decks={self.num_decks}, cards_remaining={len(self.cards)})"

2. Now design your game on a UML diagram. You may want to create classes to represent, players, a hand, and/or the game. As you work through the lab, update your UML diagram. At the end of the lab, submit your diagram (as pdf file) along with your notebook. 

3. Begin with implementing the skeleton (ie define data members and methods/functions, but do not code the logic) of the classes in your UML diagram.

4. Complete the implementation by coding the logic of all functions. For now, just implement the dealer player and human player.

5.  Test. Demonstrate game play. For example, create a game of several dealer players and show that the game is functional through several rounds.

In [ ]:
class AutoPlayer(Player):
    """
    Temporary automated player for testing.
    Hits below 16, stands at 16 or more.
    This allows us to demonstrate gameplay without user input.
    """
    def play_turn(self, deck, visible_cards):
        print(f"\n{self.name}'s turn:")
        while self.hand.get_total() < 16:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            print(f"{self.name} hits and draws {card}. Total = {self.hand.get_total()}")

            if self.hand.is_bust():
                print(f"{self.name} busts!")
                break

        if not self.hand.is_bust():
            print(f"{self.name} stands at {self.hand.get_total()}")


def play_round(deck, dealer, players):
    print("\n==================== NEW ROUND ====================")

    # Reset hands
    dealer.reset_hand()
    for p in players:
        p.reset_hand()

    # Initial deal: 2 cards each
    for _ in range(2):
        for p in players:
            card, plastic = deck.draw_card()
            p.hand.add_card(card)
        card, plastic = deck.draw_card()
        dealer.hand.add_card(card)

    # Show initial state
    print("\nDealer shows:", dealer.hand.cards[0])
    for p in players:
        print(f"{p.name} has: {p.hand.cards} (total = {p.hand.get_total()})")

    # Players take turns
    visible_cards = []  # In full game, this would track all face-up cards
    for p in players:
        p.play_turn(deck, visible_cards)

    # Dealer plays
    print("\nDealer's turn:")
    print("Dealer's hand:", dealer.hand.cards, "(total =", dealer.hand.get_total(), ")")
    dealer.play_turn(deck)
    print("Dealer final hand:", dealer.hand.cards, "(total =", dealer.hand.get_total(), ")")

    # Determine outcomes
    dealer_total = dealer.hand.get_total()
    dealer_bust = dealer.hand.is_bust()

    print("\n----- RESULTS -----")
    for p in players:
        player_total = p.hand.get_total()
        if p.hand.is_bust():
            print(f"{p.name} loses (busted).")
        elif dealer_bust:
            print(f"{p.name} wins (dealer bust).")
        elif player_total > dealer_total:
            print(f"{p.name} wins ({player_total} vs {dealer_total}).")
        elif player_total < dealer_total:
            print(f"{p.name} loses ({player_total} vs {dealer_total}).")
        else:
            print(f"{p.name} pushes (tie).")


deck = Deck(num_decks=2)  # smaller shoe for testing
dealer = Dealer()

players = [
    AutoPlayer("Alice", chips=100),
    AutoPlayer("Bob", chips=100),
    AutoPlayer("Charlie", chips=100)
]

# Play several rounds
for _ in range(3):
    play_round(deck, dealer, players)

6. Implement a new player with the following strategy:

    * Assign each card a value: 
        * Cards 2 to 6 are +1 
        * Cards 7 to 9 are 0 
        * Cards 10 through Ace are -1
    * Compute the sum of the values for all cards seen so far.
    * Hit if sum is very negative, stay if sum is very positive. Select a threshold for hit/stay, e.g. 0 or -2.  

In [ ]:
import random
from enum import Enum

class Suit(Enum):
    CLUBS = "C"
    DIAMONDS = "D"
    HEARTS = "H"
    SPADES = "S"

class Rank(Enum):
    TWO = "2"
    THREE = "3"
    FOUR = "4"
    FIVE = "5"
    SIX = "6"
    SEVEN = "7"
    EIGHT = "8"
    NINE = "9"
    TEN = "10"
    JACK = "J"
    QUEEN = "Q"
    KING = "K"
    ACE = "A"

class Card:
    def __init__(self, rank, suit, is_plastic=False):
        self.rank = rank
        self.suit = suit
        self.is_plastic = is_plastic

    def __repr__(self):
        if self.is_plastic:
            return "<PLASTIC>"
        return f"{self.rank.value}{self.suit.value}"

# DECK / SHOE

class Deck:
    def __init__(self, num_decks=6):
        self.num_decks = num_decks
        self.cards = []
        self._build_shoe()
        self._insert_plastic_card()
        self.shuffle()

    def _build_shoe(self):
        for _ in range(self.num_decks):
            for suit in Suit:
                for rank in Rank:
                    self.cards.append(Card(rank, suit))

    def _insert_plastic_card(self):
        total = len(self.cards)
        start = total // 2
        end = int(total * 0.9)
        index = random.randint(start, end)
        self.cards.insert(index, Card(None, None, is_plastic=True))
        self.plastic_index = index

    def shuffle(self):
        plastic = [c for c in self.cards if c.is_plastic]
        self.cards = [c for c in self.cards if not c.is_plastic]
        random.shuffle(self.cards)
        self._insert_plastic_card()

    def draw_card(self):
        card = self.cards.pop(0)
        return card, card.is_plastic

    def cards_remaining(self):
        return len(self.cards)
# HAND

class Hand:
    def __init__(self):
        self.cards = []

    def add_card(self, card):
        self.cards.append(card)

    def get_total(self):
        total = 0
        aces = 0

        for card in self.cards:
            if card.is_plastic:
                continue
            if card.rank in (Rank.JACK, Rank.QUEEN, Rank.KING, Rank.TEN):
                total += 10
            elif card.rank == Rank.ACE:
                aces += 1
                total += 11
            else:
                total += int(card.rank.value)

        while total > 21 and aces > 0:
            total -= 10
            aces -= 1

        return total

    def is_bust(self):
        return self.get_total() > 21

    def is_blackjack(self):
        return len(self.cards) == 2 and self.get_total() == 21

# PLAYER BASE CLASS

class Player:
    def __init__(self, name, chips=100):
        self.name = name
        self.chips = chips
        self.hand = Hand()

    def place_bet(self):
        return 10

    def play_turn(self, deck, visible_cards):
        raise NotImplementedError

    def reset_hand(self):
        self.hand = Hand()


# DEALER

class Dealer:
    def __init__(self):
        self.hand = Hand()

    def play_turn(self, deck):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)

    def reset_hand(self):
        self.hand = Hand()


# AUTO PLAYER (for testing)

class AutoPlayer(Player):
    def play_turn(self, deck, visible_cards):
        print(f"\n{self.name}'s turn:")
        while self.hand.get_total() < 16:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            print(f"{self.name} hits and draws {card}. Total = {self.hand.get_total()}")
            if self.hand.is_bust():
                print(f"{self.name} busts!")
                break
        if not self.hand.is_bust():
            print(f"{self.name} stands at {self.hand.get_total()}")


# COUNTING PLAYER (Hi-Lo Strategy)

class CountingPlayer(Player):
    def __init__(self, name, chips=100, threshold=0):
        super().__init__(name, chips)
        self.running_count = 0
        self.threshold = threshold

    def update_count(self, card):
        if card.is_plastic:
            return
        if card.rank in (Rank.TWO, Rank.THREE, Rank.FOUR, Rank.FIVE, Rank.SIX):
            self.running_count += 1
        elif card.rank in (Rank.SEVEN, Rank.EIGHT, Rank.NINE):
            self.running_count += 0
        else:
            self.running_count -= 1

    def play_turn(self, deck, visible_cards):
        print(f"\n{self.name}'s turn (running count = {self.running_count})")

        for card in visible_cards:
            self.update_count(card)

        while True:
            total = self.hand.get_total()
            print(f"{self.name}'s hand: {self.hand.cards} (total = {total})")

            if self.running_count < self.threshold:
                action = "hit"
            else:
                action = "stand"

            print(f"{self.name} chooses to {action} (count = {self.running_count})")

            if action == "stand":
                break

            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            self.update_count(card)
            print(f"{self.name} draws {card}")

            if self.hand.is_bust():
                print(f"{self.name} busts!")
                break


# GAMEPLAY DEMO

def play_round(deck, dealer, players):
    print("\n==================== NEW ROUND ====================")

    dealer.reset_hand()
    for p in players:
        p.reset_hand()

    for _ in range(2):
        for p in players:
            card, plastic = deck.draw_card()
            p.hand.add_card(card)
        card, plastic = deck.draw_card()
        dealer.hand.add_card(card)

    print("\nDealer shows:", dealer.hand.cards[0])
    for p in players:
        print(f"{p.name} has: {p.hand.cards} (total = {p.hand.get_total()})")

    visible_cards = [dealer.hand.cards[0]]
    for p in players:
        p.play_turn(deck, visible_cards)

    print("\nDealer's turn:")
    print("Dealer's hand:", dealer.hand.cards, "(total =", dealer.hand.get_total(), ")")
    dealer.play_turn(deck)
    print("Dealer final hand:", dealer.hand.cards, "(total =", dealer.hand.get_total(), ")")

    dealer_total = dealer.hand.get_total()
    dealer_bust = dealer.hand.is_bust()

    print("\n----- RESULTS -----")
    for p in players:
        player_total = p.hand.get_total()
        if p.hand.is_bust():
            print(f"{p.name} loses (busted).")
        elif dealer_bust:
            print(f"{p.name} wins (dealer bust).")
        elif player_total > dealer_total:
            print(f"{p.name} wins ({player_total} vs {dealer_total}).")
        elif player_total < dealer_total:
            print(f"{p.name} loses ({player_total} vs {dealer_total}).")
        else:
            print(f"{p.name} pushes (tie).")


# RUN DEMO

deck = Deck(num_decks=2)
dealer = Dealer()

players = [
    AutoPlayer("Alice"),
    CountingPlayer("Counter1", threshold=-2),
    CountingPlayer("Counter2", threshold=0)
]

for _ in range(3):
    play_round(deck, dealer, players)

7. Create a test scenario where one player, using the above strategy, is playing with a dealer and 3 other players that follow the dealer's strategy. Each player starts with same number of chips. Play 50 rounds (or until the strategy player is out of money). Compute the strategy player's winnings. You may remove unnecessary printouts from your code (perhaps implement a verbose/quiet mode) to reduce the output.

In [ ]:
import random
from enum import Enum

class Suit(Enum):
    CLUBS = "C"
    DIAMONDS = "D"
    HEARTS = "H"
    SPADES = "S"

class Rank(Enum):
    TWO = "2"
    THREE = "3"
    FOUR = "4"
    FIVE = "5"
    SIX = "6"
    SEVEN = "7"
    EIGHT = "8"
    NINE = "9"
    TEN = "10"
    JACK = "J"
    QUEEN = "Q"
    KING = "K"
    ACE = "A"

class Card:
    def __init__(self, rank, suit, is_plastic=False):
        self.rank = rank
        self.suit = suit
        self.is_plastic = is_plastic

    def __repr__(self):
        if self.is_plastic:
            return "<PLASTIC>"
        return f"{self.rank.value}{self.suit.value}"


# DECK / SHOE

class Deck:
    def __init__(self, num_decks=6):
        self.num_decks = num_decks
        self.cards = []
        self._build_shoe()
        self._insert_plastic_card()
        self.shuffle()

    def _build_shoe(self):
        for _ in range(self.num_decks):
            for suit in Suit:
                for rank in Rank:
                    self.cards.append(Card(rank, suit))

    def _insert_plastic_card(self):
        total = len(self.cards)
        start = total // 2
        end = int(total * 0.9)
        index = random.randint(start, end)
        self.cards.insert(index, Card(None, None, is_plastic=True))
        self.plastic_index = index

    def shuffle(self):
        plastic = [c for c in self.cards if c.is_plastic]
        self.cards = [c for c in self.cards if not c.is_plastic]
        random.shuffle(self.cards)
        self._insert_plastic_card()

    def draw_card(self):
        card = self.cards.pop(0)
        return card, card.is_plastic

    def cards_remaining(self):
        return len(self.cards)
# HAND

class Hand:
    def __init__(self):
        self.cards = []

    def add_card(self, card):
        self.cards.append(card)

    def get_total(self):
        total = 0
        aces = 0

        for card in self.cards:
            if card.is_plastic:
                continue
            if card.rank in (Rank.JACK, Rank.QUEEN, Rank.KING, Rank.TEN):
                total += 10
            elif card.rank == Rank.ACE:
                aces += 1
                total += 11
            else:
                total += int(card.rank.value)

        while total > 21 and aces > 0:
            total -= 10
            aces -= 1

        return total

    def is_bust(self):
        return self.get_total() > 21

    def is_blackjack(self):
        return len(self.cards) == 2 and self.get_total() == 21


# PLAYER BASE CLASS

class Player:
    def __init__(self, name, chips=100):
        self.name = name
        self.chips = chips
        self.hand = Hand()

    def place_bet(self):
        return 10

    def play_turn(self, deck, visible_cards):
        raise NotImplementedError

    def reset_hand(self):
        self.hand = Hand()


# DEALER

class Dealer:
    def __init__(self):
        self.hand = Hand()

    def play_turn(self, deck):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)

    def reset_hand(self):
        self.hand = Hand()


# DEALER-STRATEGY PLAYER (hit < 17, stand ≥ 17)

class DealerStrategyPlayer(Player):
    def play_turn(self, deck, visible_cards):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            if self.hand.is_bust():
                break


# COUNTING PLAYER (Hi-Lo Strategy)

class CountingPlayer(Player):
    def __init__(self, name, chips=100, threshold=0):
        super().__init__(name, chips)
        self.running_count = 0
        self.threshold = threshold

    def update_count(self, card):
        if card.is_plastic:
            return
        if card.rank in (Rank.TWO, Rank.THREE, Rank.FOUR, Rank.FIVE, Rank.SIX):
            self.running_count += 1
        elif card.rank in (Rank.SEVEN, Rank.EIGHT, Rank.NINE):
            pass
        else:
            self.running_count -= 1

    def play_turn(self, deck, visible_cards):
        for card in visible_cards:
            self.update_count(card)

        while True:
            total = self.hand.get_total()
            if self.running_count < self.threshold:
                action = "hit"
            else:
                action = "stand"

            if action == "stand":
                break

            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            self.update_count(card)

            if self.hand.is_bust():
                break

# GAME ROUND

def play_round(deck, dealer, players, quiet=True):
    dealer.reset_hand()
    for p in players:
        p.reset_hand()

    # Initial deal
    for _ in range(2):
        for p in players:
            card, plastic = deck.draw_card()
            p.hand.add_card(card)
        card, plastic = deck.draw_card()
        dealer.hand.add_card(card)

    visible_cards = [dealer.hand.cards[0]]

    # Players act
    for p in players:
        p.play_turn(deck, visible_cards)

    # Dealer acts
    dealer.play_turn(deck)

    # Resolve bets
    dealer_total = dealer.hand.get_total()
    dealer_bust = dealer.hand.is_bust()

    results = {}
    for p in players:
        bet = p.place_bet()
        player_total = p.hand.get_total()

        if p.hand.is_bust():
            p.chips -= bet
            results[p.name] = -bet
        elif dealer_bust:
            p.chips += bet
            results[p.name] = bet
        elif player_total > dealer_total:
            p.chips += bet
            results[p.name] = bet
        elif player_total < dealer_total:
            p.chips -= bet
            results[p.name] = -bet
        else:
            results[p.name] = 0

    return results


# 50-ROUND SIMULATION

deck = Deck(num_decks=6)
dealer = Dealer()

starting_chips = 200

counter = CountingPlayer("Counter", chips=starting_chips, threshold=-1)
others = [
    DealerStrategyPlayer("P1", chips=starting_chips),
    DealerStrategyPlayer("P2", chips=starting_chips),
    DealerStrategyPlayer("P3", chips=starting_chips)
]

players = [counter] + others

rounds = 50
for r in range(rounds):
    if counter.chips <= 0:
        break
    play_round(deck, dealer, players, quiet=True)

# Final result
strategy_winnings = counter.chips - starting_chips
strategy_winnings

8. Create a loop that runs 100 games of 50 rounds, as setup in previous question, and store the strategy player's chips at the end of the game (aka "winnings") in a list. Histogram the winnings. What is the average winnings per round? What is the standard deviation. What is the probabilty of net winning or lossing after 50 rounds?


In [ ]:
import random
from enum import Enum
import numpy as np
import matplotlib.pyplot as plt

# CARD + ENUMS

class Suit(Enum):
    CLUBS = "C"
    DIAMONDS = "D"
    HEARTS = "H"
    SPADES = "S"

class Rank(Enum):
    TWO = "2"
    THREE = "3"
    FOUR = "4"
    FIVE = "5"
    SIX = "6"
    SEVEN = "7"
    EIGHT = "8"
    NINE = "9"
    TEN = "10"
    JACK = "J"
    QUEEN = "Q"
    KING = "K"
    ACE = "A"

class Card:
    def __init__(self, rank, suit, is_plastic=False):
        self.rank = rank
        self.suit = suit
        self.is_plastic = is_plastic

    def __repr__(self):
        if self.is_plastic:
            return "<PLASTIC>"
        return f"{self.rank.value}{self.suit.value}"


# DECK / SHOE

class Deck:
    def __init__(self, num_decks=6):
        self.num_decks = num_decks
        self.cards = []
        self._build_shoe()
        self._insert_plastic_card()
        self.shuffle()

    def _build_shoe(self):
        for _ in range(self.num_decks):
            for suit in Suit:
                for rank in Rank:
                    self.cards.append(Card(rank, suit))

    def _insert_plastic_card(self):
        total = len(self.cards)
        start = total // 2
        end = int(total * 0.9)
        index = random.randint(start, end)
        self.cards.insert(index, Card(None, None, is_plastic=True))
        self.plastic_index = index

    def shuffle(self):
        plastic = [c for c in self.cards if c.is_plastic]
        self.cards = [c for c in self.cards if not c.is_plastic]
        random.shuffle(self.cards)
        self._insert_plastic_card()

    def draw_card(self):
        card = self.cards.pop(0)
        return card, card.is_plastic

    def cards_remaining(self):
        return len(self.cards)


# HAND

class Hand:
    def __init__(self):
        self.cards = []

    def add_card(self, card):
        self.cards.append(card)

    def get_total(self):
        total = 0
        aces = 0

        for card in self.cards:
            if card.is_plastic:
                continue
            if card.rank in (Rank.JACK, Rank.QUEEN, Rank.KING, Rank.TEN):
                total += 10
            elif card.rank == Rank.ACE:
                aces += 1
                total += 11
            else:
                total += int(card.rank.value)

        while total > 21 and aces > 0:
            total -= 10
            aces -= 1

        return total

    def is_bust(self):
        return self.get_total() > 21


# PLAYER BASE CLASS

class Player:
    def __init__(self, name, chips=100):
        self.name = name
        self.chips = chips
        self.hand = Hand()

    def place_bet(self):
        return 10

    def play_turn(self, deck, visible_cards):
        raise NotImplementedError

    def reset_hand(self):
        self.hand = Hand()


# DEALER

class Dealer:
    def __init__(self):
        self.hand = Hand()

    def play_turn(self, deck):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)

    def reset_hand(self):
        self.hand = Hand()


# DEALER-STRATEGY PLAYER

class DealerStrategyPlayer(Player):
    def play_turn(self, deck, visible_cards):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            if self.hand.is_bust():
                break


# COUNTING PLAYER (Hi-Lo Strategy)

class CountingPlayer(Player):
    def __init__(self, name, chips=100, threshold=-1):
        super().__init__(name, chips)
        self.running_count = 0
        self.threshold = threshold

    def update_count(self, card):
        if card.is_plastic:
            return
        if card.rank in (Rank.TWO, Rank.THREE, Rank.FOUR, Rank.FIVE, Rank.SIX):
            self.running_count += 1
        elif card.rank in (Rank.SEVEN, Rank.EIGHT, Rank.NINE):
            pass
        else:
            self.running_count -= 1

    def play_turn(self, deck, visible_cards):
        for card in visible_cards:
            self.update_count(card)

        while True:
            if self.running_count < self.threshold:
                action = "hit"
            else:
                action = "stand"

            if action == "stand":
                break

            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            self.update_count(card)

            if self.hand.is_bust():
                break

# GAME ROUND

def play_round(deck, dealer, players):
    dealer.reset_hand()
    for p in players:
        p.reset_hand()

    # Initial deal
    for _ in range(2):
        for p in players:
            card, plastic = deck.draw_card()
            p.hand.add_card(card)
        card, plastic = deck.draw_card()
        dealer.hand.add_card(card)

    visible_cards = [dealer.hand.cards[0]]

    # Players act
    for p in players:
        p.play_turn(deck, visible_cards)

    # Dealer acts
    dealer.play_turn(deck)

    # Resolve bets
    dealer_total = dealer.hand.get_total()
    dealer_bust = dealer.hand.is_bust()

    for p in players:
        bet = p.place_bet()
        player_total = p.hand.get_total()

        if p.hand.is_bust():
            p.chips -= bet
        elif dealer_bust:
            p.chips += bet
        elif player_total > dealer_total:
            p.chips += bet
        elif player_total < dealer_total:
            p.chips -= bet


# RUN 100 GAMES OF 50 ROUNDS

def run_single_game():
    deck = Deck(num_decks=6)
    dealer = Dealer()

    starting_chips = 200
    counter = CountingPlayer("Counter", chips=starting_chips, threshold=-1)
    others = [
        DealerStrategyPlayer("P1", chips=starting_chips),
        DealerStrategyPlayer("P2", chips=starting_chips),
        DealerStrategyPlayer("P3", chips=starting_chips)
    ]
    players = [counter] + others

    for _ in range(50):
        if counter.chips <= 0:
            break
        play_round(deck, dealer, players)

    return counter.chips - starting_chips


# Run 100 games
results = [run_single_game() for _ in range(100)]

# ANALYSIS

avg_winnings = np.mean(results)
std_dev = np.std(results)
prob_win = np.mean([1 if r > 0 else 0 for r in results])
prob_loss = np.mean([1 if r < 0 else 0 for r in results])

print("Average winnings per 50-round game:", avg_winnings)
print("Average winnings per round:", avg_winnings / 50)
print("Standard deviation:", std_dev)
print("Probability of net winning:", prob_win)
print("Probability of net losing:", prob_loss)

# HISTOGRAM

plt.hist(results, bins=15, edgecolor='black')
plt.title("Distribution of Counting Player Winnings (100 games)")
plt.xlabel("Net winnings after 50 rounds")
plt.ylabel("Frequency")
plt.show()

9. Repeat previous questions scanning the value of the threshold. Try at least 5 different threshold values. Can you find an optimal value?

In [ ]:
import random
from enum import Enum
import numpy as np
import matplotlib.pyplot as plt

class Suit(Enum):
    CLUBS = "C"
    DIAMONDS = "D"
    HEARTS = "H"
    SPADES = "S"

class Rank(Enum):
    TWO = "2"
    THREE = "3"
    FOUR = "4"
    FIVE = "5"
    SIX = "6"
    SEVEN = "7"
    EIGHT = "8"
    NINE = "9"
    TEN = "10"
    JACK = "J"
    QUEEN = "Q"
    KING = "K"
    ACE = "A"

class Card:
    def __init__(self, rank, suit, is_plastic=False):
        self.rank = rank
        self.suit = suit
        self.is_plastic = is_plastic

class Deck:
    def __init__(self, num_decks=6):
        self.num_decks = num_decks
        self.cards = []
        self._build_shoe()
        self._insert_plastic_card()
        self.shuffle()

    def _build_shoe(self):
        for _ in range(self.num_decks):
            for suit in Suit:
                for rank in Rank:
                    self.cards.append(Card(rank, suit))

    def _insert_plastic_card(self):
        total = len(self.cards)
        start = total // 2
        end = int(total * 0.9)
        index = random.randint(start, end)
        self.cards.insert(index, Card(None, None, is_plastic=True))
        self.plastic_index = index

    def shuffle(self):
        plastic = [c for c in self.cards if c.is_plastic]
        self.cards = [c for c in self.cards if not c.is_plastic]
        random.shuffle(self.cards)
        self._insert_plastic_card()

    def draw_card(self):
        card = self.cards.pop(0)
        return card, card.is_plastic


class Hand:
    def __init__(self):
        self.cards = []

    def add_card(self, card):
        self.cards.append(card)

    def get_total(self):
        total = 0
        aces = 0

        for card in self.cards:
            if card.is_plastic:
                continue
            if card.rank in (Rank.JACK, Rank.QUEEN, Rank.KING, Rank.TEN):
                total += 10
            elif card.rank == Rank.ACE:
                aces += 1
                total += 11
            else:
                total += int(card.rank.value)

        while total > 21 and aces > 0:
            total -= 10
            aces -= 1

        return total

    def is_bust(self):
        return self.get_total() > 21

class Player:
    def __init__(self, name, chips=100):
        self.name = name
        self.chips = chips
        self.hand = Hand()

    def place_bet(self):
        return 10

    def play_turn(self, deck, visible_cards):
        raise NotImplementedError

    def reset_hand(self):
        self.hand = Hand()

class Dealer:
    def __init__(self):
        self.hand = Hand()

    def play_turn(self, deck):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)

    def reset_hand(self):
        self.hand = Hand()

class DealerStrategyPlayer(Player):
    def play_turn(self, deck, visible_cards):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            if self.hand.is_bust():
                break

class CountingPlayer(Player):
    def __init__(self, name, chips=100, threshold=-1):
        super().__init__(name, chips)
        self.running_count = 0
        self.threshold = threshold

    def update_count(self, card):
        if card.is_plastic:
            return
        if card.rank in (Rank.TWO, Rank.THREE, Rank.FOUR, Rank.FIVE, Rank.SIX):
            self.running_count += 1
        elif card.rank in (Rank.SEVEN, Rank.EIGHT, Rank.NINE):
            pass
        else:
            self.running_count -= 1

    def play_turn(self, deck, visible_cards):
        for card in visible_cards:
            self.update_count(card)

        while True:
            if self.running_count < self.threshold:
                action = "hit"
            else:
                action = "stand"

            if action == "stand":
                break

            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            self.update_count(card)

            if self.hand.is_bust():
                break

def play_round(deck, dealer, players):
    dealer.reset_hand()
    for p in players:
        p.reset_hand()

    # Initial deal
    for _ in range(2):
        for p in players:
            card, plastic = deck.draw_card()
            p.hand.add_card(card)
        card, plastic = deck.draw_card()
        dealer.hand.add_card(card)

    visible_cards = [dealer.hand.cards[0]]

    # Players act
    for p in players:
        p.play_turn(deck, visible_cards)

    # Dealer acts
    dealer.play_turn(deck)

    # Resolve bets
    dealer_total = dealer.hand.get_total()
    dealer_bust = dealer.hand.is_bust()

    for p in players:
        bet = p.place_bet()
        player_total = p.hand.get_total()

        if p.hand.is_bust():
            p.chips -= bet
        elif dealer_bust:
            p.chips += bet
        elif player_total > dealer_total:
            p.chips += bet
        elif player_total < dealer_total:
            p.chips -= bet


def run_single_game(threshold):
    deck = Deck(num_decks=6)
    dealer = Dealer()

    starting_chips = 200
    counter = CountingPlayer("Counter", chips=starting_chips, threshold=threshold)
    others = [
        DealerStrategyPlayer("P1", chips=starting_chips),
        DealerStrategyPlayer("P2", chips=starting_chips),
        DealerStrategyPlayer("P3", chips=starting_chips)
    ]
    players = [counter] + others

    for _ in range(50):
        if counter.chips <= 0:
            break
        play_round(deck, dealer, players)

    return counter.chips - starting_chips

threshold_values = [-3, -2, -1, 0, 1]
results_by_threshold = {}

for t in threshold_values:
    results = [run_single_game(t) for _ in range(100)]
    results_by_threshold[t] = results

    plt.hist(results, bins=15, edgecolor='black')
    plt.title(f"Winnings Distribution (threshold={t})")
    plt.xlabel("Net winnings after 50 rounds")
    plt.ylabel("Frequency")
    plt.show()

    print(f"\n=== Threshold {t} ===")
    print("Average winnings:", np.mean(results))
    print("Average per round:", np.mean(results) / 50)
    print("Std deviation:", np.std(results))
    print("Prob winning:", np.mean([1 if r > 0 else 0 for r in results]))
    print("Prob losing:", np.mean([1 if r < 0 else 0 for r in results]))

10. Create a new strategy based on web searches or your own ideas. Demonstrate that the new strategy will result in increased or decreased winnings. 

In [ ]:
import random
from enum import Enum
import numpy as np
import matplotlib.pyplot as plt

class Suit(Enum):
    CLUBS = "C"
    DIAMONDS = "D"
    HEARTS = "H"
    SPADES = "S"

class Rank(Enum):
    TWO = "2"
    THREE = "3"
    FOUR = "4"
    FIVE = "5"
    SIX = "6"
    SEVEN = "7"
    EIGHT = "8"
    NINE = "9"
    TEN = "10"
    JACK = "J"
    QUEEN = "Q"
    KING = "K"
    ACE = "A"

class Card:
    def __init__(self, rank, suit, is_plastic=False):
        self.rank = rank
        self.suit = suit
        self.is_plastic = is_plastic

class Deck:
    def __init__(self, num_decks=6):
        self.num_decks = num_decks
        self.cards = []
        self._build_shoe()
        self._insert_plastic_card()
        self.shuffle()

    def _build_shoe(self):
        for _ in range(self.num_decks):
            for suit in Suit:
                for rank in Rank:
                    self.cards.append(Card(rank, suit))

    def _insert_plastic_card(self):
        total = len(self.cards)
        start = total // 2
        end = int(total * 0.9)
        index = random.randint(start, end)
        self.cards.insert(index, Card(None, None, is_plastic=True))
        self.plastic_index = index

    def shuffle(self):
        plastic = [c for c in self.cards if c.is_plastic]
        self.cards = [c for c in self.cards if not c.is_plastic]
        random.shuffle(self.cards)
        self._insert_plastic_card()

    def draw_card(self):
        card = self.cards.pop(0)
        return card, card.is_plastic

class Hand:
    def __init__(self):
        self.cards = []

    def add_card(self, card):
        self.cards.append(card)

    def get_total(self):
        total = 0
        aces = 0

        for card in self.cards:
            if card.is_plastic:
                continue
            if card.rank in (Rank.JACK, Rank.QUEEN, Rank.KING, Rank.TEN):
                total += 10
            elif card.rank == Rank.ACE:
                aces += 1
                total += 11
            else:
                total += int(card.rank.value)

        while total > 21 and aces > 0:
            total -= 10
            aces -= 1

        return total

    def is_bust(self):
        return self.get_total() > 21

class Player:
    def __init__(self, name, chips=100):
        self.name = name
        self.chips = chips
        self.hand = Hand()

    def place_bet(self):
        return 10

    def play_turn(self, deck, visible_cards):
        raise NotImplementedError

    def reset_hand(self):
        self.hand = Hand()

class Dealer:
    def __init__(self):
        self.hand = Hand()

    def play_turn(self, deck):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)

    def reset_hand(self):
        self.hand = Hand()

class DealerStrategyPlayer(Player):
    def play_turn(self, deck, visible_cards):
        while self.hand.get_total() < 17:
            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            if self.hand.is_bust():
                break

class CountingPlayer(Player):
    def __init__(self, name, chips=100, threshold=-1):
        super().__init__(name, chips)
        self.running_count = 0
        self.threshold = threshold

    def update_count(self, card):
        if card.is_plastic:
            return
        if card.rank in (Rank.TWO, Rank.THREE, Rank.FOUR, Rank.FIVE, Rank.SIX):
            self.running_count += 1
        elif card.rank in (Rank.SEVEN, Rank.EIGHT, Rank.NINE):
            pass
        else:
            self.running_count -= 1

    def play_turn(self, deck, visible_cards):
        for card in visible_cards:
            self.update_count(card)

        while True:
            if self.running_count < self.threshold:
                action = "hit"
            else:
                action = "stand"

            if action == "stand":
                break

            card, plastic = deck.draw_card()
            self.hand.add_card(card)
            self.update_count(card)

            if self.hand.is_bust():
                break

class HighCardHunter(Player):
    """
    New strategy:
    - If dealer shows 2–6 → stand earlier (stand at 12+)
    - If dealer shows 7–A → hit until 17
    - If player's hand has many low cards → hit more
    - If player's hand has many high cards → stand earlier
    """

    def play_turn(self, deck, visible_cards):
        dealer_up = visible_cards[0]

        def count_low(cards):
            return sum(1 for c in cards if c.rank in 
                       (Rank.TWO, Rank.THREE, Rank.FOUR, Rank.FIVE, Rank.SIX))

        def count_high(cards):
            return sum(1 for c in cards if c.rank in 
                       (Rank.TEN, Rank.JACK, Rank.QUEEN, Rank.KING, Rank.ACE))

        while True:
            total = self.hand.get_total()
            lows = count_low(self.hand.cards)
            highs = count_high(self.hand.cards)

            # Dealer weak → stand early
            if dealer_up.rank in (Rank.TWO, Rank.THREE, Rank.FOUR, Rank.FIVE, Rank.SIX):
                if total >= 12 or highs >= 2:
                    break

            # Dealer strong → hit aggressively
            if dealer_up.rank in (Rank.SEVEN, Rank.EIGHT, Rank.NINE, Rank.TEN, Rank.JACK, Rank.QUEEN, Rank.KING, Rank.ACE):
                if total >= 17 or highs >= 3:
                    break

            # Otherwise hit
            card, plastic = deck.draw_card()
            self.hand.add_card(card)

            if self.hand.is_bust():
                break

def play_round(deck, dealer, players):
    dealer.reset_hand()
    for p in players:
        p.reset_hand()

    # Initial deal
    for _ in range(2):
        for p in players:
            card, plastic = deck.draw_card()
            p.hand.add_card(card)
        card, plastic = deck.draw_card()
        dealer.hand.add_card(card)

    visible_cards = [dealer.hand.cards[0]]

    # Players act
    for p in players:
        p.play_turn(deck, visible_cards)

    # Dealer acts
    dealer.play_turn(deck)

    # Resolve bets
    dealer_total = dealer.hand.get_total()
    dealer_bust = dealer.hand.is_bust()

    for p in players:
        bet = p.place_bet()
        player_total = p.hand.get_total()

        if p.hand.is_bust():
            p.chips -= bet
        elif dealer_bust:
            p.chips += bet
        elif player_total > dealer_total:
            p.chips += bet
        elif player_total < dealer_total:
            p.chips -= bet

def run_single_game(strategy_class):
    deck = Deck(num_decks=6)
    dealer = Dealer()

    starting_chips = 200
    strat_player = strategy_class("Strategy", chips=starting_chips)
    others = [
        DealerStrategyPlayer("P1", chips=starting_chips),
        DealerStrategyPlayer("P2", chips=starting_chips),
        DealerStrategyPlayer("P3", chips=starting_chips)
    ]
    players = [strat_player] + others

    for _ in range(50):
        if strat_player.chips <= 0:
            break
        play_round(deck, dealer, players)

    return strat_player.chips - starting_chips


counting_results = [run_single_game(lambda name, chips=200: CountingPlayer(name, chips, threshold=-1)) for _ in range(100)]
hunter_results = [run_single_game(HighCardHunter) for _ in range(100)]



def analyze(results, label):
    print(f"\n=== {label} ===")
    print("Average winnings:", np.mean(results))
    print("Average per round:", np.mean(results) / 50)
    print("Std deviation:", np.std(results))
    print("Prob winning:", np.mean([1 if r > 0 else 0 for r in results]))
    print("Prob losing:", np.mean([1 if r < 0 else 0 for r in results]))

    plt.hist(results, bins=15, edgecolor='black')
    plt.title(f"Winnings Distribution: {label}")
    plt.xlabel("Net winnings after 50 rounds")
    plt.ylabel("Frequency")
    plt.show()

analyze(counting_results, "Counting Strategy")
analyze(hunter_results, "High-Card Hunter Strategy")